(ch:classification)=
# 분류

:::{note} 감사의 글

오렐리앙 제롱<font size='2'>Aurélien Géron</font>의 [Hands-On Machine Learning with Scikit-Learn and PyTorch (O'Reilly, 2025)](https://github.com/ageron/handson-mlp)에 사용된 코드를 참고한 강의노트이다. 보다 심화된 이해를 위해 책 원본을 읽을 것을 강력하게 권장한다. 자료를 공개한 오렐리앙 제롱과 일부 그림 자료를 제공해 준 한빛아카데미에게 진심어린 감사를 전한다.
:::

:::{seealso} 코드 실행
[(코드 워크아웃) 분류](https://colab.research.google.com/github/codingalzi/code-workout-ml/blob/master/notebooks/code-classification.ipynb)를 병행하여 읽을 것을 권장한다.
:::

MNIST 손글씨 데이터셋을 이용하여 이진 분류와 다중 클래스 분류 모델을 훈련하고, 혼동 행렬과 정밀도·재현율을 통한 성능 평가 및 오류 분석 방법을 다룬다.

## MNIST 데이터셋

미국의 고등학생과 인구조사국 직원들이 손으로 직접 쓴 70,000개의 0부터 9 사이의 정수 숫자 손글씨 이미지로 구성된 데이터셋이다.

### 데이터 기초 정보

**입력 데이터셋과 타깃 데이터셋**

입력 데이터 샘플은 손글씨 이미지 데이터를 표현한 `(28, 28)` 모양의 넘파이 2차원 어레이를 `28x28=784` 크기의 1차원 어레이로 변환한 값이다.
각 손글씨 이미지에 대한 라벨은 정수가 아니라 `'0'`, `'1'`, ..., `'9'`처럼 문자열로 지정되어 있다.

| 구분 | 변수명 | 설명 |
| :--- | :--- | :--- |
| **입력 데이터셋** | `X` | `28x28=784` 크기의 1차원 어레이로 제공되는 손글씨 이미지 데이터 샘플 7만개로 구성된 넘파이 2차원 어레이 (모양: `(70000, 784)`) |
| **타깃 데이터셋** | `y` | 각 손글씨 이미지에 해당하는 정답 라벨이 `'0'`부터 `'9'`까지의 문자열로 지정된 넘파이 1차원 어레이 (모양: `(70000, )`) |

:::{note} 이미지 픽셀 데이터

모델 훈련에 사용되는 입력 데이터셋의 모든 샘플은 길이가 784인 1차원 어레이로 주어졌다.
어레이의 각 항목은 `(28, 28)` 모양의 손글씨 이미지에 포함된 하나의 픽셀(화소)값에 해당한다.
즉, 이미지에 포함된 모든 픽셀이 입력 데이터 샘플의 특성이 된다.
픽셀값은 0과 255 사이의 정수이며, 0은 완전히 검은색을, 255는 완전히 흰색을 가리킨다.
그리고 정수가 커질수록 밝은 색을 가리킨다.

예를 들어 첫째 샘플을 `(28, 28)` 모양의 2차원 어레이로 나타내면 다음과 같다.

<div align="center"><img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch03/mnist_digit_5-array.png?raw=true" width="1000"/></div>
:::

위 어레이에서 0이 아닌 정수들의 형태를 따라 그리면 정수 5처럼 그려진다.
실제로 위 어레이를 이미지로 출력하면 아래와 같으며, 샘플의 실제 라벨도 정수 5로 지정되어 있다.

<div align="center"><img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch03/mnist_digit_5.jpg?raw=true" width="300"/></div>

아래 이미지는 100개의 손글씨 이미지를 동시에 보여준다.
손글씨를 쓴 사람들의 개성이 드러나며 일부 손글씨는 정확히 어떤 정수를 표현하는지 판단하기 어렵다.

<div align="center"><img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch03/homl03-01.png?raw=true" width="450"/></div>

### 머신러닝 훈련 모델 선택

손글씨 이미지가 표현하는 0부터 9까지의 정수를 타깃으로 삼아 예측하는 모델을 훈련시키려 한다.

* 지도학습: 각 이미지가 담고 있는 숫자가 라벨, 즉 정답으로 지정되어 있다.
* 모델: 손글씨 이미지 데이터를 분석하여 0부터 9까지, 각 손글씨 이미지를 총 10개의 범주로
    분류해야 한다.
    이런 경우 **다중 클래스 분류**<font size='2'>multiclass classification</font> 모델을 활용해야 한다.

### 훈련셋과 테스트셋

입력 데이터셋 `X`와 `y`는 이미 6:1 의 비율로 훈련셋과 테스트셋으로 분류되어 있다.
또한 10개의 숫자에 대해 균등하게 무작위로 잘 섞여 있다.

| 구분 | 변수명 | 설명 |
| :--- | :--- | :--- |
| 훈련 입력 데이터셋 | `X_train` | 앞쪽 60,000개 이미지 데이터로 구성된 `(60000, 784)` 모양의 넘파이 2차원 어레이 |
| 훈련 타깃 데이터셋 | `y_train` | 6만개의 라벨로 구성된 `(60000, )` 모양의 넘파이 1차원 어레이 |
| 테스트 입력 데이터셋 | `X_test` | 나머지 1만개의 이미지 데이터로 구성된 `(10000, 784)` 모양의 넘파이 2차원 어레이 |
| 테스트 타깃 데이터셋 | `y_test` | 1만개의 라벨로 구성된 `(10000, )` 모양의 넘파이 1차원 어레이 |

## 이진 분류기: 숫자-5 감별기

10개의 범주(클래스)를 구분하는 다중 클래스 모델을 다루기에 앞서, 손글씨 이미지가 숫자 5인지 여부만 판별하는 **이진 분류기**<font size='2'>binary classifier</font>를 먼저 훈련한다. 이를 통해 분류 모델의 기본적인 훈련 과정과 성능 평가 방식을 직관적으로 살펴본다.

이 '숫자-5 감별기'를 훈련하기 위해, 기존 타깃 데이터셋을 숫자 5인 경우는 `1`(양성), 5가 아닌 다른 숫자인 경우는 `0`(음성)으로 지정한 새로운 타깃 데이터셋(`y_train_5`)으로 변환한다.

| `y_train_5`의 라벨 | 설명 | 비율 |
| :---: | :--- | :--- |
| `1` | 숫자 5를 가리키는 이미지의 라벨 (양성) | 약 10% |
| `0` | 숫자 5 이외의 수를 가리키는 이미지의 라벨 (음성) | 약 90% |

훈련을 위한 이진 분류기 모델로는 `SGDClassifier`를 활용한다. 확률적 경사하강법<font size='2'>stochastic gradient descent</font> 분류기라고도 불리는 이 모델은 한 번에 하나씩 훈련 샘플을 처리하여 속도가 매우 빠르기 때문에 대용량 데이터셋 처리에 유용한 선형 모델이다. 구체적인 훈련 방식은 다음 장에서 다룬다.

```python
sgd_clf = SGDClassifier(random_state=42)
sgd_clf.fit(X_train, y_train_5)
```

위 코드를 실행하면 `SGDClassifier` 모델이 훈련 데이터(`X_train`)와 타깃 데이터셋(`y_train_5`)을 분석하여, 입력된 픽셀 이미지에서 어떤 패턴이 나타날 때 숫자 5에 해당하는지 그 규칙을 학습한다.

## 이진 분류기 성능 측정

이진 분류기를 포함하여 훈련이 완료된 모든 분류기의 성능 평가는 일반적으로 다음 세 가지를 활용하여 진행한다.

* 정확도
* 정밀도와 재현율
* ROC 곡선의 AUC

여기서는 정확도, 정밀도, 재현율을 상세하게 소개한다.
ROC 곡선의 AUC에 대해서는 설명하지 않는 대신 [분류: ROC 및 AUC](https://developers.google.com/machine-learning/crash-course/classification/roc-and-auc?hl=ko)를 참고할 것을 추천한다.

분류기의 성능 평가 세 가지 방식 모두 혼동 행렬을 활용한다.

### 혼동 행렬

**혼동 행렬**<font size="2">confusion matrix</font>은 클래스별 예측 결과를 정리한 행렬이다.
훈련이 완료된 이진 분류기인 숫자-5 감별기(`sgd_clf`)에 대한 혼동 행렬은 아래와 같은 (2, 2) 모양의 2차원 (넘파이) 어레이로 생성된다.

```python
array([[53892,   687],
       [ 1891,  3530]])
```

:::{prf:example} 혼동 행렬 예제
:label: exp_confusion_matrix

아래 그림은 혼동 행렬에 포함된 각 정수의 의미를 설명하기 위해
숫자-5 감별기의 판별 결과를 단순화해서 보여준다. 

<p><div align="center"><img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch03/homl03-02.png?raw=true" width="500"/></div></p>

아래 표는 위 그림에 포함된 각 칸을 가리키는 용어와 칸에 포함된 손글씨 이미지들에 대해 설명한다.

| 실제 라벨 | 예측: 음성 (5 아니라고 판별) | 예측: 양성 (5라고 판별) |
| :--- | :--- | :--- |
| 음성<br>(5 아님) | TN (참 음성, True Negative)<br>5 아닌 숫자를 5가 아니라고 예측<br>(예: 8, 3, 9, 7, 2) | FP (거짓 양성, False Positive)<br>5 아닌 숫자를 5라고 예측<br>(예: 6) |
| 양성<br>(5 맞음) | FN (거짓 음성, False Negative)<br>실제 5인 숫자를 5가 아니라고 예측 | TP (참 양성, True Positive)<br>실제 5인 숫자를 5라고 예측 |

위 단순화된 그림을 토대로한 혼동 행렬은 다음과 같이 작성된다.

```python
array([[5, 1],
       [2, 3]])
```
:::

### 정확도

**정확도**<font size='2'>accuracy</font>는 실제 라벨을 정확하게 예측한 비율이다.
훈련된 SGD 분류기(`sgd_clf`)의 정확도는 96% 정도로 매우 높게 계산된다.

```
sgd_clf_accuracy = (TP + TN)/(TP+FP+TN+FN)
                 = (3530 + 53892)/(3530 + 687 + 53892 + 1891)
                 = 0.957
```

**정확도의 한계**

정확도가 96% 정도로 매우 좋은 결과로 보인다. 하지만 "무조건 5가 아니다" 라고 예측하는 모델도 90%의 정확도를 보인다. 이유는 6만개의 훈련셋에 0부터 9까지의 손글씨 이미지가 균등하게 포함되어 있어서, 5가 아닌 다른 숫자를 표현하는 손글씨가 전체의 90% 정도를 차지하기 때문이다.

특정 범주에 속하는 데이터가 상대적으로 너무 많을 경우 정확도는 신뢰하기 어렵다.
이런 경우엔 정확도 보다는 정밀도와 재현율을 이용한 평가 방식이 권장된다.

### 정밀도와 재현율

**정밀도**<font size="2">precision</font>

모델이 양성으로 예측한 결과의 정확도를 나타낸다. 우리 예제에서는 분류기가 숫자 5라고 예측한 이미지 중에서 실제로 숫자 5인 이미지의 비율을 의미한다.

```python
sgd_clf_precision = TP / (TP + FP) = 3530 / (3530 + 687) = 0.837
```

**재현율**<font size='2'>recall</font>

실제 양성 샘플을 모델이 얼마나 잘 찾아냈는지를 나타내는 지표이며, **참 양성 비율**<font size="2">true positive rate (TPR)</font>이라고도 부른다. 즉, 실제 숫자 5인 전체 이미지 중에서 분류기가 5라고 올바르게 감지해낸 이미지의 비율을 가리킨다.

```python
sgd_clf_recall = TP / (TP + FN) = 3530 / (3530 + 1891) = 0.651
```

**정밀도와 재현율의 상대적 중요도**

모델 사용 목적에 따라 최소화해야 하는 오류(오분류)의 종류가 다를 수 있으며, 이에 따라 정밀도와 재현율 중 어떤 지표를 우선시할지 결정해야 한다.

| 강조 지표 | 핵심 목표 (최소화할 오류) | 주요 예시 | 지표별 의미와 상대적 중요도 |
| :--- | :--- | :--- | :--- |
| 재현율<br>(Recall) | 거짓 음성(False Negative) 최소화<br><font size="2">진짜 양성을 누락하면 치명적인 경우</font> | 암 진단<br>금융 사기 탐지 | • 재현율 (중요): 실제 암을 암으로 진단한 사례의 비율<br>• 정밀도: 암이라고 판별된 사례 중 실제 암인 사례의 비율 |
| 정밀도<br>(Precision) | 거짓 양성(False Positive) 최소화<br><font size="2">가짜 양성이 결과에 섞여들면 곤란한 경우</font> | 건전한 영상 선택<br>스팸 메일 필터 | • 정밀도 (중요): '건전'하다고 판별받은 영상 중 실제 건전한 영상의 비율<br>• 재현율: 실제 건전한 영상 중 모델이 맞게 찾아낸 비율 |

### 정밀도와 재현율의 상호 반비례 관계

**결정 함수와 결정 임곗값**

훈련된 이진 분류기는 개별 샘플의 범주를 구분하기 위해 점수를 계산하는 **결정 함수**<font size="2">decision function</font>를 제공한다. 분류기는 이 함수가 계산한 점수가 **결정 임곗값**<font size="2">decision threshold</font> 이상이면 양성으로, 그보다 작으면 음성으로 판별한다.

예를 들어 `SGDClassifier`는 `decision_function()` 메서드를 활용하여, 결정 함숫값이 0보다 작으면 음성, 0 이상이면 양성으로 분류한다.
훈련된 숫자-5 감별기(`sgd_clf`)에 처음 10개 샘플을 넣어 결정 함숫값을 확인해보면 다음과 같다. 첫 번째 샘플의 점수만 양수이고 나머지 9개는 모두 음수이므로, 첫 번째 샘플만 '5'로 판별되고 나머지 9개는 '5가 아니다'라고 판별된다.

```python
array([  1200.93051237, -26883.79202424, -33072.03475406, -15919.5480689 ,
       -20003.53970191, -16652.87731528, -14276.86944263, -23328.13728948,
        -5172.79611432, -13873.5025381 ])
```

**트레이드오프 관계**

정밀도와 재현율은 서로 반비례하는 트레이드오프<font size="2">tradeoff</font> 관계를 갖는다. 한쪽 지표가 증가하면 다른 쪽 지표는 감소하므로, 모델의 목적에 맞게 두 지표 사이의 적절한 균형점을 찾아야 한다. 이 정밀도와 재현율의 균형을 조절하는 핵심 요소가 바로 모델의 **결정 임곗값**이다.

결정 임곗값의 변화에 따라 두 지표가 어떻게 달라지는지 아래 예제에서 확인할 수 있다.

* 결정 임곗값이 커질수록: 모델이 양성으로 판별하는 기준이 엄격해져 정밀도는 올라가지만, 그만큼 놓치는 양성 샘플이 많아져 재현율은 떨어진다.
* 결정 임곗값이 작아질수록: 모델이 양성으로 판별하는 기준이 느슨해져 재현율은 올라가지만, 잘못된 양성 예측(거짓 양성)도 함께 늘어나 정밀도는 떨어진다.

:::{prf:example} 정밀도와 재현율의 트레이드오프
:label: exp_decision_threshold

아래 그림에서 세 개의 화살표 (a), (b), (c)는 서로 다른 결정 임곗값을 가리키며, 
화살표 윗쪽에 위치한 정밀도와 재현율은 해당 결정 임곗값을 기준으로
주어진 샘플의 양성, 음성 여부를 판단할 경우의 정밀도와 재현율이다. 

<div align="center">
    <img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch03/homl03-03.png?raw=true" width="500"/>
</div>

| 경우 | 정밀도 | 재현율 |
| :---: | :--- | :--- |
| (a) | 80% (양성 예측 5개 중 실제 5인 샘플 4개, 아닌 샘플 1개) | 67% (실제 5인 샘플 총 6개 중 5라고 판별된 샘플 4개) |
| (b) | 75% (양성 예측 8개 중 실제 5인 샘플 6개, 아닌 샘플 2개) | 100% (실제 5인 샘플 총 6개 중 5라고 판별된 샘플 6개) |
| (c) | 100% (양성 예측 3개 중 실제 5인 샘플 3개, 아닌 샘플 0개) | 50% (실제 5인 샘플 총 6개 중 5라고 판별된 샘플 3개) |

:::

**결정 임곗값/정밀도/재현율 그래프**

아래 그래프는 숫자-5 감별기 `sgd_clf`가 계산한 샘플들의 결정 함숫값을 바탕으로, "결정 임곗값 변화에 따른 정밀도와 재현율의 추이"를 보여준다. 

그래프를 보면 결정 임곗값을 높일 때 재현율은 꾸준히 감소하는 반면, 정밀도는 전반적으로는 상승하지만 특정 구간에서 일시적으로 떨어지는 현상이 나타날 수도 있다. 하지만 임곗값이 계속 커지면 결과적으로 정밀도는 1.0(100%)에 도달한다.

<p><div align="center"><img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch03/homl03-04.png?raw=true" width="500"/></div></p>

**정밀도/재현율 그래프**

위 그래프를 재현율 대 정밀도 그래프로 변환하면 다음과 같다.
결정 임곗값(threshold)을 낮춰 재현율을 올리면 정밀도는 떨어지는
상호 반비례 관계를 잘 보여준다.

<div align="center"><img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch03/homl03-05.png?raw=true" width="400"/></div>

**정밀도 90% 분류기 구현**

위 그래프에서 검은색 수직 점선은 정밀도는 90%, 재현율은 50% 정도가 되게 하는 결정 임곗값의
위치를 표시하며, 실제 값은 3,000이다.
결정 임곗값을 변경하여 원하는 정밀도와 재현율을 갖는 숫자-5 감별기를 구현하는 방법은
[정밀도 90%분류기 구현](https://colab.research.google.com/github/codingalzi/code-workout-ml/blob/master/notebooks/code-classification.ipynb#scrollTo=cKfpQLyuCHkf)에서 자세히 소개한다.

## 다중 클래스 분류

**다중 클래스 분류**<font size="2">multiclass classification</font>는 세 개 이상의 범주(클래스) 중 하나로 샘플을 분류한다.
예를 들어, MNIST 손글씨 숫자 문제는 입력값이 주어질 때 이를 0부터 9까지 총 10개의 범주로 분류하도록 다중 클래스 모델을 훈련하는 작업이다.

### 다중 클래스 분류 지원 모델

아래에 언급된 모델들을 포함한 많은 모델이 이진 분류와 다중 클래스 분류를 모두 지원한다.

* `LogisticRegression` 모델
* `RandomForestClassifier` 모델
* `SGDClassifier` 모델

**다중 클래스 분류 모델 교차 검증**

숫자-5 감별기와는 다르게, 0부터 9까지의 범주를 다루는 다중 클래스 분류 문제에서는 각 클래스(숫자)에 해당하는 데이터가 고르게 분포되어 있으므로 정확도를 기준으로 모델의 성능을 평가해도 무방하다.

예를 들어, `SGDClassifier` 모델에 대해 교차 검증을 수행하여 정확도를 계산하면 약 86.7% 수준으로 확인된다.

**다중 클래스 분류의 정밀도와 재현율**

데이터가 불균형하거나 특정 범주의 오분류를 엄격하게 통제해야 할 경우 다중 클래스 모델에서도 정밀도와 재현율을 주요 평가지표로 활용한다. 다중 클래스 환경에서는 앞서 살펴본 '숫자-5 감별기'와 같은 이진 분류 방식을 각 범주마다 적용하는 **일대다(One-vs-Rest)** 방식을 주로 활용한다. 즉, 평가하고자 하는 특정 범주만을 양성(Positive)으로, 그 외의 나머지 모든 범주를 묶어 음성(Negative)으로 간주하는 여러 개의 이진 분류 문제를 통해 개별 범주들의 정밀도와 재현율을 먼저 각각 계산한다.

이후 도출된 개별 범주의 지표들을 다중 클래스 모델 전체의 단일 평가 점수로 통합하기 위해 주로 다음과 같은 평균(Average) 방식을 사용한다.

* **매크로 평균(Macro average)**: 각 범주에 속한 샘플 수와 무관하게 모든 범주의 정밀도/재현율을 단순히 더해 평균을 낸다. 모든 범주를 동등하게 취급하므로, 데이터가 적은 소수 범주에 대한 모델의 예측 성능을 중요하게 다룰 때 적합하다.
* **가중 평균(Weighted average)**: 개별 범주의 지표에 해당 범주의 실제 샘플 수(support)를 가중치로 곱하여 평균을 계산한다. 데이터셋의 실제 분포 비율을 반영하여 모델의 전반적인 성능을 평가할 때 유리하다.

:::{note} 참고 자료
다중 클래스 분류의 평가지표 계산 방식(`average` 매개변수)에 대한 더 자세한 수학적 정의와 설명은 [Scikit-Learn 공식 문서: 다중 클래스 및 다중 라벨 분류 평가](https://scikit-learn.org/stable/modules/model_evaluation.html#multiclass-and-multilabel-classification)를 참고한다.
:::

**스케일링의 중요성**

입력 데이터에 표준화 스케일링 전처리를 적용하면 모델의 예측 정확도가 89.7%까지 향상된다.
머신러닝 모델의 훈련은 기본적으로 정수 연산보다는 부동소수점, 그것도 단위가 작고 값들의 분포가 작을 때 보다 성능이 좋아진다.
그런데 표준화 대신 Min-Max 스케일링을 적용하면 성능이 91.05%까지 더욱 개선된다.
손글씨 이미지의 픽셀은 원래 모두 양수값을 갖는다. 하지만 표준화 스케일링을 거치면 일부 픽셀값이 음수로 변환되는데, 이러한 변화가 모델의 훈련 성능에 악영향을 미치는 것으로 추정된다.

### 다중 클래스 분류기 모델의 오류 분석

그리드 탐색(Grid Search)이나 랜덤 탐색(Randomized Search) 등을 이용한 모델 미세조정(Fine-Tuning) 과정을 거쳐 최선의 모델을 찾았다고 가정하자. 

이제 본격적으로 오류 분석을 진행하여 모델의 성능을 평가하고, 이를 바탕으로 추가적인 개선 방안을 모색하는 과정을 살펴본다. 이 과정에서 가장 먼저 활용되는 도구가 바로 혼동 행렬이다.

**다중 클래스 분류 모델의 혼동 행렬**

아래 왼쪽 이미지는 훈련된 다중 클래스 분류 모델의 혼동 행렬을 색상으로 시각화한 결과다. 주대각선 상의 색상이 전반적으로 밝은 것은 모델의 분류가 대체로 정확하게 이루어졌음을 의미한다. 다만, 5번과 8번 행의 색상이 다른 곳보다 상대적으로 어두운데, 이는 모델이 '숫자 5'와 '숫자 8'을 분류하는 정확도가 상대적으로 낮다는 것을 보여준다.

반면에 아래 오른쪽 이미지는 오분류의 패턴을 더 명확히 파악하기 위해 혼동 행렬의 값을 각 숫자의 전체 개수 대비 비율로 변환한 결과다. 즉, 각 행(실제 숫자)에 속한 값들의 합이 100%가 되도록 정규화(Normalization)하였다.

<p><div align="center"><img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch03/homl03-08.png?raw=true" width="100%"/></div></p>

**오차율 활용**

위 오른쪽 이미지를 보면 손글씨 이미지를 '숫자 8' 또는 '숫자 9'로 잘못 판별한 비율이 가장 높다. 즉, 8번과 9번 열의 오분류 비율 합이 다른 열에 비해 상대적으로 크다.

올바르게 예측된 샘플을 제외하고 행(실제 범주)을 기준으로 오분류 비율만 시각화하면 아래 왼쪽 이미지와 같다. 8번과 9번 열이 전반적으로 밝게 나타나며, 이는 많은 손글씨 이미지가 8 또는 9로 잘못 판별되었음을 의미한다. 

아래 오른쪽 이미지는 열(예측 범주)을 기준으로 오분류 데이터를 정규화한 결과를 보여준다. 예를 들어, 7로 오인된 전체 손글씨 이미지 중에서 실제 숫자 9인 이미지의 비율이 41%에 달한다는 점을 확인할 수 있다.

<p><div align="center"><img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch03/homl03-09.png?raw=true" width="100%"/></div></p>

**개별 오류 확인**

위 오른쪽 이미지를 보면 숫자 5로 잘못 판별된 이미지 중에서 실제 숫자가 3인 이미지의 비율이 29%로 상당히 높다는 것을 알 수 있다. 

구체적인 오분류 양상을 확인하기 위해 숫자 3과 5 데이터에 대해서만 예측 결과를 혼동 행렬 형태로 시각화하면 아래와 같다. 여기서 행은 실제 범주(라벨)를, 열은 예측 범주(라벨)를 나타낸다.

<div align="center"><img src="https://github.com/codingalzi/code-workout-ml/blob/master/images/ch03/homl03-10.png?raw=true" width="450"/></div>

**데이터 증식**

사람의 눈으로 보아도 3과 5를 명확히 구분하기 어려운 손글씨가 있다. 특히 이번 예제에서 사용한 SGD 분류기는 단순한 선형 모델이므로 이러한 픽셀 패턴의 미세한 차이를 학습하는 데 성능상의 한계가 있다. 분류 성능을 높이기 위해 더 복잡한 모델을 도입할 수도 있지만, 근본적으로는 훈련 데이터의 양을 늘리는 것이 필요하다.

그러나 새로운 손글씨 이미지를 수작업으로 추가 수집하는 것은 현실적으로 매우 어렵다. 이에 대한 대안으로, 기존 이미지를 조금씩 회전하거나 뒤집고 이동시키는 등 약간의 변형을 가하여 훈련 데이터를 인위적으로 늘릴 수 있다. 이러한 기법을 **데이터 증식**<font size='2'>data augmentation</font>이라 부른다.

일반적으로 고도화된 이미지 데이터 증식 과정은 딥러닝 라이브러리에서 제공하는 전용 도구들을 주로 활용한다. 머신러닝의 기초를 중심적으로 다루는 본 장에서는 딥러닝 모델과 관련된 상세한 설명은 생략한다.

## 연습문제

**문제 1**

[(코드 워크아웃) 분류](https://colab.research.google.com/github/codingalzi/code-workout-ml/blob/master/notebooks/code-classification.ipynb)의
본문 내용을 학습하라.

**문제 2**

MNIST 테스트셋에 대한 정확도가 97% 이상 나오는 MNIST 분류기를 훈련시켜 보아라.

힌트: [정확도 97% 성능의 MNIST 분류기](https://colab.research.google.com/github/codingalzi/code-workout-ml/blob/master/notebooks/code-classification.ipynb#scrollTo=wsPjQ28VCHkm)

**문제 3**

[Kaggle](https://www.kaggle.com/c/titanic) 타이타닉 데이터셋으로 생존 여부 분류 문제를 풀어 보아라.

힌트: [타이타닉 데이터셋 도전](https://colab.research.google.com/github/codingalzi/code-workout-ml/blob/master/notebooks/code-classification.ipynb#scrollTo=D6CIjV_ICHko)